# siRNA Dataset

### Import Libraries

In [35]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt # data visualization
import seaborn as sns # statistical data visualization


### Import Dataset

In [36]:
data = 'siRNA/data.csv'

df = pd.read_csv(data)

### Exploratory Data Analysis

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
#Columns names

col_names = df.columns

print(col_names)

In [ ]:
df.info()

In [ ]:
#Checking missing values

df.isnull().sum()

In [ ]:
# Excluding columns that include ID num (to be ıncluded only int.)
df2 = df.drop(['Target seq', 'Sequence'], axis=1)

X = df2.iloc[:, :-1]
y = df2['Activity'] 

# Display first few rows
print(X.head())
print(y.head())

# Summary statistics
print(X.describe())
print(y.describe())

## Black Box Algorithms

#### Import Libraries

* Requirements: **tensorflow**

In [64]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

### Deep Neural Network

#### Feature Scaling

In [93]:
# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on the entire dataset and transform
X_scaled = scaler.fit_transform(X)

#### Split Dataset

In [ ]:
# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

#### Model 1

In [ ]:
# Define the model architecture
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='linear')  # Output layer for regression
])

# Display the model's architecture
model.summary()

In [96]:
model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mean_squared_error']
)

In [ ]:
# Define Early Stopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate on test data
test_loss, test_mse = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Mean Squared Error (MSE): {test_mse:.4f}")

# Predict on test data
y_pred = model.predict(X_test).flatten()

# Calculate additional metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Test R-squared (R²): {r2:.4f}")

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(history.history['mean_squared_error'], label='Training MSE')
plt.plot(history.history['val_mean_squared_error'], label='Validation MSE')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Training and Validation MSE Over Epochs')
plt.legend()
plt.show()